In [1]:
from langchain_chroma import Chroma
from langchain_classic.retrievers import SelfQueryRetriever
from langchain_classic.chains.query_constructor.schema import AttributeInfo
from models import get_model, get_embeddings
from pdf_chunk import text_spliter


e:\Rohanta_AI_workbook\adavance_RAG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
llm= get_model()
embedding_model = get_embeddings()
chunks = text_spliter
chroma_vectorstore=Chroma.from_documents(chunks ,embedding_model )

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3295.80it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
# ── Step 2: Define what metadata fields exist ──
metadata_field_info = [
    AttributeInfo(
        name="source",
        description="The source or author of the document",
        type="string",
    ),
    AttributeInfo(
        name="year",
        description="The year the document was written or published",
        type="integer",
    ),
    AttributeInfo(
        name="topic",
        description="The main topic or category of the document",
        type="string",
    ),
]

In [4]:
# ── Step 3: Describe what the documents contain ──
document_content_description = "Professional resume and career information about Rohanta Bhamare"


In [17]:
# ── Step 4: Build the Self Query Retriever ──
self_query_retriever = SelfQueryRetriever.from_llm(
    llm=llm,
    vectorstore=chroma_vectorstore,
    document_contents=document_content_description,
    metadata_field_info=metadata_field_info,
    verbose=True,        # shows what filter the LLM generated
    enable_limit=True,   # lets you say "give me 3 results"
    search_kwargs={"k": 10}
)

In [18]:
docs = self_query_retriever.invoke("What is Rohanta's current university?")

In [19]:
for i in docs:
    print(i)

page_content='CAREER OBJECTIVE' metadata={'source': 'my_data.txt'}
page_content='Secondary Education:
SSC (Secondary School Certificate): 82%
HSC (Higher Secondary Certificate): 57%


COMPLETE TECHNICAL SKILLS' metadata={'source': 'my_data.txt'}
page_content='ENGINEERING PHILOSOPHY' metadata={'source': 'my_data.txt'}
page_content='Degree: Master of Engineering (ME)
Institution: Savitribai Phule Pune University, Pune, India
Duration: 2017 to 2020
Grade: First Class with Distinction, CGPA 8
Thesis: Sentiment Analysis on Customer Reviews. Built NLP pipelines using TF-IDF, classical ML models, and LSTM networks. Deployed via Flask API.

Degree: Bachelor of Engineering (BE)
Institution: Savitribai Phule Pune University, Pune, India
Duration: 2013 to 2017
Grade: First Class with Distinction, 70%' metadata={'source': 'my_data.txt'}
page_content='EDUCATION

Degree: MSc in Artificial Intelligence
Institution: Berlin School of Business and Innovation (BSBI), Berlin, Germany
Duration: March 2026 

In [20]:
# ── Step 6: Plug into LCEL chain ──
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

In [21]:
prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the context below.

Context: {context}
Question: {question}
""")

In [22]:
def format_docs(docs):
    return "\n\n".join([d.page_content for d in docs])

In [23]:
chain = (
    {
        "context": self_query_retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [24]:
print(chain.invoke("What is Rohanta's current university?"))

Rohanta is currently attending the Berlin School of Business and Innovation (BSBI) in Berlin, Germany, pursuing his MSc in Artificial Intelligence.
